In [1]:
# Dependensi unit disediakan oleh lingkungan luring yang dikunci.
# Tidak ada instalasi paket pada saat pembaca dijalankan.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

from scipy.linalg import expm
from scipy.stats import binom


In [3]:
α = 0.6
λ = 0.5
γ = 0.1
b = 10


In [4]:
@njit
def draw_X(T, X_0, max_iter=5000):
    """
    Bangkitkan satu realisasi X_T dengan syarat X_0.
    """

    J, Y = 0.0, X_0
    m = 0

    while m < max_iter:
        s = 1 / γ if Y == 0 else 1 / λ
        # W ~ Exp(γ) pada keadaan 0; W ~ Exp(λ) pada keadaan lainnya.
        W = np.random.exponential(scale=s)
        J += W
        if J >= T:
            return Y
        # Jika belum mencapai T, perbarui Y.
        if Y == 0:
            Y = b
        else:
            U = np.random.geometric(α)
            Y = Y - min(Y, U)
        m += 1

    raise RuntimeError("max_iter tercapai sebelum waktu T")


@njit
def independent_draws(T=10, num_draws=100, seed=20260824):
    "Bangkitkan vektor realisasi X_T yang saling bebas."

    np.random.seed(seed)
    draws = np.empty(num_draws, dtype=np.int64)

    for i in range(num_draws):
        X_0 = np.random.binomial(b, 0.25)
        draws[i] = draw_X(T, X_0)

    return draws


In [5]:
T = 30
n = b + 1
states = np.arange(n)
draws = independent_draws(T, num_draws=100_000, seed=20260824)
prob_empiris = np.array([np.mean(draws == i) for i in states])

fig, ax = plt.subplots()
ax.bar(states, prob_empiris, width=0.8, alpha=0.6)
ax.set_xlabel("persediaan", fontsize=14)
ax.set_ylabel("probabilitas empiris", fontsize=14)
ax.set_title("Distribusi persediaan hasil simulasi pada T = 30")
fig.set_label(
    "Diagram batang probabilitas empiris persediaan untuk keadaan 0 sampai 10."
)

print("keadaan,probabilitas_empiris")
for state, probability in zip(states, prob_empiris):
    print(f"{state},{probability:.12f}")

plt.show()


In [6]:
α = 0.6
λ = 0.5
γ = 0.1
b = 10
T = 30
n = b + 1
states = np.arange(n)
I = np.identity(n)

# Matriks rantai lompatan tertanam
K = np.zeros((n, n))
K[0, -1] = 1
for i in range(1, n):
    for j in range(0, i):
        if j == 0:
            K[i, j] = (1 - α)**(i-1)
        else:
            K[i, j] = α * (1 - α)**(i-j-1)

# Intensitas lompatan sebagai fungsi keadaan
r = np.ones(n) * λ
r[0] = γ

# Matriks Q
Q = np.empty_like(K)
for i in range(n):
    for j in range(n):
        Q[i, j] = r[i] * (K[i, j] - I[i, j])


def P_t(ψ, t):
    return ψ @ expm(t * Q)


ψ_0 = binom.pmf(states, b, 0.25)
ψ_T = P_t(ψ_0, T)

fig, ax = plt.subplots()
ax.bar(states, ψ_T, width=0.8, alpha=0.6)
ax.set_xlabel("persediaan", fontsize=14)
ax.set_ylabel("probabilitas", fontsize=14)
ax.set_title("Distribusi persediaan eksak pada T = 30")
fig.set_label(
    "Diagram batang probabilitas eksak persediaan untuk keadaan 0 sampai 10."
)

print("keadaan,probabilitas_eksak")
for state, probability in zip(states, ψ_T):
    print(f"{state},{probability:.12f}")

plt.show()
